# 🎙️ Fish Audio Drama 3 (35B-A3B) + Fish Speech WebUI в Google Colab (A100 GPU)

Оптимизированный ноутбук «всё в одном» для запуска флагманской модели синтеза речи, клонирования голоса и многоголосой озвучки сцен **[Fish Audio Drama 3](https://huggingface.co/fishaudio/drama-3)** с официальным веб-интерфейсом **[Fish Speech Awesome WebUI](https://github.com/fishaudio/fish-speech)** на видеокарте **NVIDIA A100 (40 ГБ / 80 ГБ VRAM)**.

### 💡 Ключевые особенности этого шаблона
- **⚡ Эксклюзивная A100 W8A16 FP8-MoE оптимизация**:
  - По умолчанию `fish-speech` на видеокартах A100 (архитектура SM80) распаковывает все 35 млрд параметров MoE из FP8 в BF16 при старте, что требует ~72 ГБ видеопамяти и вызывает ошибку нехватки памяти (OOM) на стандартной A100 40 ГБ.
  - Наш ноутбук автоматически применяет **W8A16 FP8-MoE патч**: все 256 экспертов остаются в видеопамяти в компактном формате FP8 (**~36 ГБ VRAM**), а в BF16 на лету декодируются только активные эксперты (`3B` параметров на токен). Благодаря этому модель стабильно работает **как на A100 40 ГБ, так и на A100 80 ГБ**!
- **🖥️ Официальный интерфейс Fish Speech Awesome WebUI**:
  - Автоматическая сборка современного TypeScript-интерфейса (`awesome_webui`) и привязка его к корневому адресу сервера (`/` и `/ui`).
- **🎭 Возможности модели Fish Audio Drama 3**:
  - **Многоголосые сцены до 16 персонажей** в одном тексте с помощью тегов `<|speaker:0|>` … `<|speaker:15|>`.
  - **Управление эмоциями и интонацией** прямо в тексте: `[sighs]`, `[laughs]`, `[whisper]`, `[angry]`, `[crying]`, `[excited]`, `[pause]`.
  - **Zero-shot клонирование голоса** на ~80 языках (включая русский, английский, японский и др.) по короткому образцу 5–30 сек.
  - **Точечное редактирование и режиссёрские правки** (`previous_takes`) без перегенерации с нуля.
- **💾 Полная интеграция с Google Drive**:
  - Веса модели (`~38 ГБ`) кэшируются в `MyDrive/FishSpeech_Models/drama-3` (повторное скачивание не требуется).
  - Ваши референсные голоса сохраняются в `MyDrive/FishSpeech_References`, а сгенерированные записи — в `MyDrive/FishSpeech_Outputs`.
- **🚀 3 быстрых способа подключения к WebUI (без тормозов Cloudflare)**:
  - **Google Colab Direct Proxy**, **Pinggy Tunnel** и **Cloudflare Tunnel (HTTP/2)** выдаются одновременно.

> ⚠️ **Внимание**: Перед запуском убедитесь, что в верхнем меню **«Среда выполнения» (Runtime) → «Сменить среду выполнения» (Change runtime type)** выбран ускоритель **A100 GPU**.

## Step 1: Проверка окружения (A100 GPU) и подключение Google Drive

In [ ]:
# Проверяем выделенный GPU и объём видеопамяти (A100 40GB или 80GB)
!nvidia-smi

# Сохранение кэша модели, референсных голосов и результатов на Google Drive
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive успешно подключён!")

## Step 2: Установка Fish Speech, сборка Awesome WebUI и применение A100 FP8-MoE патча

- Устанавливает аудио-библиотеки (`portaudio`, `sox`, `ffmpeg`) и зависимости `fishaudio/fish-speech`.
- Собирает официальный фронтенд **Awesome WebUI**.
- Применяет оптимизацию **W8A16 FP8-MoE для A100 (SM80)**, снижающую потребление VRAM в 2 раза.

In [ ]:
import os
from pathlib import Path

# @title Настройки оптимизации под A100
ENABLE_A100_FP8_MOE_PATCH = True  # @param {type:"boolean"}
DISABLE_CUDA_GRAPH_FOR_40GB = True  # @param {type:"boolean"}

print("📦 Установка системных аудио-пакетов и Node.js...")
!apt-get update -qq && apt-get install -y -qq portaudio19-dev libsox-dev ffmpeg nodejs npm

%cd /content
if not os.path.exists("/content/fish-speech"):
    !git clone https://github.com/fishaudio/fish-speech.git
else:
    %cd /content/fish-speech
    !git pull

%cd /content/fish-speech
print("⚡ Установка Python-зависимостей Fish Speech (с сохранением оптимизированного PyTorch Colab)...")
!pip install -q "huggingface_hub[cli]" "transformers>=5.0.0" "tokenizers>=0.21.0" \
    "hydra-core>=1.3.2" "natsort>=8.4.0" "einops>=0.7.0" "librosa>=0.10.1" \
    "kui>=1.6.0" "uvicorn>=0.30.0" "loguru>=0.6.0" "pyrootutils>=1.0.4" \
    "resampy>=0.4.3" "einx[torch]==0.2.2" pydub pyaudio "modelscope>=1.17.1" \
    "opencc-python-reimplemented==0.1.7" silero-vad ormsgpack "tiktoken>=0.8.0" \
    "pydantic>=2.9.2" cachetools descript-audio-codec safetensors "soundfile>=0.12.1" \
    "protobuf>=3.20.0,<6.0.0" loralib
!pip install -q --no-deps -e .

# Сборка официального веб-интерфейса Awesome WebUI
webui_dist = Path("/content/fish-speech/awesome_webui/dist/index.html")
if not webui_dist.exists():
    print("🛠️ Сборка интерфейса Fish Speech Awesome WebUI...")
    %cd /content/fish-speech/awesome_webui
    !npm install --silent --no-audit --no-fund
    !npm run build
    %cd /content/fish-speech
print(f"✅ WebUI собран: {webui_dist.exists()}")

# Патч 1: Открытие WebUI сразу по корневому адресу '/' (а также '/ui')
views_path = Path("/content/fish-speech/tools/server/views.py")
views_code = views_path.read_text(encoding="utf-8")
if '@routes.http("/")' not in views_code:
    views_code = views_code.replace(
        '@routes.http("/ui")\nclass WebUI(HttpView):',
        '@routes.http("/")\n@routes.http("/ui")\nclass WebUI(HttpView):'
    )
    views_path.write_text(views_code, encoding="utf-8")
    print("✅ Корневой маршрут '/' привязан к Awesome WebUI.")

# Патч 2: W8A16 FP8-MoE для A100 (SM80) — хранит 35B экспертов в FP8 (~36 ГБ VRAM вместо 72 ГБ)
if ENABLE_A100_FP8_MOE_PATCH:
    fp8_path = Path("/content/fish-speech/fish_speech/models/text2semantic/fish_omni/fp8_moe.py")
    moe_path = Path("/content/fish-speech/fish_speech/models/text2semantic/fish_omni/moe.py")

    if fp8_path.exists() and moe_path.exists():
        fp8_code = fp8_path.read_text(encoding="utf-8")
        fp8_code = fp8_code.replace(
            "torch.cuda.get_device_capability(device) >= (8, 9)",
            "torch.cuda.get_device_capability(device) >= (8, 0)"
        )
        fp8_path.write_text(fp8_code, encoding="utf-8")

        moe_code = moe_path.read_text(encoding="utf-8")
        moe_code = moe_code.replace(
            "if self.fp8 and x.is_cuda:",
            "if self.fp8 and x.is_cuda and torch.cuda.get_device_capability(x.device) >= (8, 9):"
        )
        moe_path.write_text(moe_code, encoding="utf-8")
        print("🚀 Применён патч A100 W8A16 FP8-MoE (экономия ~35 ГБ VRAM на A100)!")

print("✅ Установка и настройка Fish Speech завершены!")

## Step 3: Загрузка весов `fishaudio/drama-3` и привязка папок Google Drive

- Если `USE_GOOGLE_DRIVE = True`:
  - **Кэш весов Drama 3**: `/content/drive/MyDrive/FishSpeech_Models/drama-3` (скачивается только при первом запуске)
  - **Референсные голоса**: `/content/drive/MyDrive/FishSpeech_References` (все загруженные в WebUI голоса сохраняются навсегда)
  - **Папка для готовых аудио**: `/content/drive/MyDrive/FishSpeech_Outputs`

In [ ]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

# Опционально: укажите HF_TOKEN, если скачиваете на высокой скорости с авторизацией
HF_TOKEN = ""  # @param {type:"string"}
if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()

os.makedirs("/content/fish-speech/checkpoints", exist_ok=True)

if USE_GOOGLE_DRIVE:
    DRIVE_MODELS_DIR = "/content/drive/MyDrive/FishSpeech_Models/drama-3"
    DRIVE_REFS_DIR = "/content/drive/MyDrive/FishSpeech_References"
    DRIVE_OUT_DIR = "/content/drive/MyDrive/FishSpeech_Outputs"
    os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
    os.makedirs(DRIVE_REFS_DIR, exist_ok=True)
    os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

    # Симлинк для чекпоинта drama-3
    local_ckpt = "/content/fish-speech/checkpoints/drama-3"
    if os.path.exists(local_ckpt) and not os.path.islink(local_ckpt):
        !rm -rf {local_ckpt}
    if not os.path.exists(local_ckpt):
        os.symlink(DRIVE_MODELS_DIR, local_ckpt)

    # Симлинк для референсных голосов
    local_refs = "/content/fish-speech/references"
    if os.path.exists(local_refs) and not os.path.islink(local_refs):
        !rm -rf {local_refs}
    if not os.path.exists(local_refs):
        os.symlink(DRIVE_REFS_DIR, local_refs)

    TARGET_CKPT_DIR = DRIVE_MODELS_DIR
    print(f"📁 Папки синхронизированы с Google Drive:\n  - Модель: {DRIVE_MODELS_DIR}\n  - Референсы: {DRIVE_REFS_DIR}\n  - Вывод: {DRIVE_OUT_DIR}")
else:
    TARGET_CKPT_DIR = "/content/fish-speech/checkpoints/drama-3"
    os.makedirs(TARGET_CKPT_DIR, exist_ok=True)
    os.makedirs("/content/fish-speech/references", exist_ok=True)

# Проверяем, все ли 15 шардов модели и codec.pth уже есть в кэше
ckpt_path = Path(TARGET_CKPT_DIR)
shards = list(ckpt_path.glob("model-*-of-00015.safetensors"))
has_codec = (ckpt_path / "codec.pth").exists()

if len(shards) == 15 and has_codec:
    print("⚡ Все 15 шардов модели fishaudio/drama-3 и codec.pth уже найдены в кэше! Скачивание пропускается.")
else:
    print("⬇️ Скачивание весов fishaudio/drama-3 с Hugging Face (~38 ГБ)...")
    snapshot_download(
        repo_id="fishaudio/drama-3",
        local_dir=TARGET_CKPT_DIR,
        local_dir_use_symlinks=False
    )
    print("✅ Загрузка fishaudio/drama-3 успешно завершена!")

## Step 4: Запуск сервера Fish Speech (Drama 3) + Awesome WebUI и 3 быстрых туннеля

Запускает `tools/api_server.py` с моделью **Drama 3** и сразу выдаёт **3 быстрые ссылки для входа в WebUI и работы по API**:
1. 🚀 **Google Colab Direct Proxy** (`*.googleusercontent.com`) — максимальная скорость напрямую через серверы Google (открывайте в том же браузере, где открыт Colab).
2. ⚡ **Pinggy Tunnel** (`https://xxxx.a.free.pinggy.link`) — быстрый SSH/TCP-туннель без Cloudflare (отлично открывается с ПК без VPN и с телефона).
3. 🌐 **Cloudflare Tunnel (HTTP/2)** (`https://xxxx.trycloudflare.com`) — резервный туннель и эндпоинт для API.

### 🎭 Шпаргалка по фишкам Drama 3 в интерфейсе WebUI:
- **Эмоции и действия прямо в тексте**: `[sighs]`, `[laughs]`, `[whisper]`, `[angry]`, `[crying]`, `[excited]`, `[pause]`.
- **Многоголосый диалог (до 16 спикеров)**: используйте теги `<|speaker:0|> Привет! <|speaker:1|> [laughs] Рад тебя слышать!`

In [ ]:
import os
import subprocess
import threading
import time
import re
import torch

# Остановим старые процессы, если ячейку перезапускают
!pkill -f "tools/api_server.py" || true
!pkill -f "cloudflared" || true

# Скачивание Cloudflared
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# Авто-настройка переменных окружения под объём памяти A100 (40GB vs 80GB)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 40
env = os.environ.copy()
env["FISH_DRAMA3_FP8_EXPERTS"] = "1"
if vram_gb < 65 or DISABLE_CUDA_GRAPH_FOR_40GB:
    # Отключаем выделение лишних буферов CUDA Graph, чтобы гарантировать запас VRAM на A100 40GB
    env["FISH_DRAMA3_CUDA_GRAPH"] = "0"

print(f"⚡ Запуск сервера Fish Speech (Drama 3) на GPU ({vram_gb:.1f} ГБ VRAM)...")
%cd /content/fish-speech
api_proc = subprocess.Popen([
    "python", "tools/api_server.py",
    "--listen", "127.0.0.1:8080",
    "--llama-checkpoint-path", "checkpoints/drama-3",
    "--device", "cuda:0"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)

server_ready = threading.Event()

def log_server():
    for line in iter(api_proc.stdout.readline, ''):
        print("[FishSpeech]", line, end="")
        if "Startup done, listening server" in line or "Uvicorn running on" in line:
            server_ready.set()

threading.Thread(target=log_server, daemon=True).start()

print("⏳ Загрузка весов Drama 3 в видеопамять и прогрев модели (обычно 1–3 минуты)...")
server_ready.wait(timeout=300)

# 1. Встроенный прокси Google Colab (максимальная скорость в том же браузере)
try:
    from google.colab.output import eval_js
    colab_url = eval_js("google.colab.kernel.proxyPort(8080)")
except Exception:
    colab_url = None

# 2. Запуск Pinggy.io SSH-туннеля (быстрый аналог без Cloudflare)
pinggy_proc = subprocess.Popen([
    "ssh", "-p", "443",
    "-R0:127.0.0.1:8080",
    "-o", "StrictHostKeyChecking=no",
    "-o", "ServerAliveInterval=30",
    "a.pinggy.io"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

pinggy_url = None
def watch_pinggy():
    global pinggy_url
    for line in iter(pinggy_proc.stdout.readline, ''):
        m = re.search(r'https://[a-zA-Z0-9-]+\.a\.free\.pinggy\.link', line)
        if m and not pinggy_url:
            pinggy_url = m.group(0)
            print(f"\n⚡ [Pinggy Tunnel] Быстрая ссылка WebUI: {pinggy_url}/ui")

threading.Thread(target=watch_pinggy, daemon=True).start()

# 3. Запуск Cloudflare Tunnel в режиме HTTP/2
tunnel_proc = subprocess.Popen([
    "/content/cloudflared", "tunnel",
    "--url", "http://127.0.0.1:8080",
    "--protocol", "http2"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

cf_url = None
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match and not cf_url:
        cf_url = match.group(0)
        time.sleep(2)
        print("\n" + "="*72)
        print("🎉 FISH SPEECH + DRAMA 3 ГОТОВ! ВЫБЕРИТЕ ЛЮБУЮ ССЫЛКУ ДЛЯ ОТКРЫТИЯ WEBUI:")
        if colab_url:
            base_colab = colab_url.rstrip('/')
            print(f"🚀 1. Google Colab Proxy (Макс. скорость): {base_colab}/ui")
        if pinggy_url:
            print(f"⚡ 2. Pinggy Tunnel (Без Cloudflare):      {pinggy_url}/ui")
        print(f"🌐 3. Cloudflare Tunnel (HTTP/2 + API):    {cf_url}/ui")
        print("="*72 + "\n")
        print("💡 Для вызовов по API (из client.py или агента) используйте базовый URL без /ui.")
        break

def drain_cf():
    for _ in iter(tunnel_proc.stdout.readline, ''):
        pass

threading.Thread(target=drain_cf, daemon=True).start()

try:
    api_proc.wait()
except KeyboardInterrupt:
    print("\n⏹️ Сервер остановлен.")
    api_proc.terminate()
    tunnel_proc.terminate()
    pinggy_proc.terminate()

## 🛑 (После завершения работы) Как гарантированно остановить списание CU

Когда закончите озвучку и тесты, **чтобы со 100% надёжностью остановить расход Compute Units (CU)**:
1. Нажмите **«Среда выполнения» (Runtime)** в верхнем меню Colab.
2. Выберите **«Отключиться и удалить среду выполнения» (Disconnect and delete runtime)**.